# Investment Clock Regime Analysis

This notebook provides a deep dive into the Investment Clock regime inference:

## Business Cycle Framework

| Phase | Economic Conditions | Outperforming Assets |
|-------|--------------------|-----------------------|
| **Recovery** | Below-trend but rebounding | Cyclicals, Value, High Yield |
| **Expansion** | Healthy growth, neutral policy | Broad equities, Small caps |
| **Slowdown** | Peak growth, decelerating | Defensives, Quality, Long bonds |
| **Contraction** | Declining activity | Safe havens, Treasuries, Low vol |

## Key Insight

We infer regime from **asset class performance patterns** (cause), NOT from benchmark allocation (effect).

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

%matplotlib inline

In [ ]:
# Load data
from src.utils.paths import data_paths, output_paths
dp = data_paths()
op = output_paths()

# Backward-compatible aliases
PROCESSED_PATH = dp.processed.root
RAW_PATH = dp.raw.root

returns_matrix = pd.read_parquet(dp.algorithms.returns)
print(f"Returns matrix: {returns_matrix.shape}")
print(f"Date range: {returns_matrix.index.min()} to {returns_matrix.index.max()}")

---
## 1. Asset Inference

First, we need to infer what assets each algorithm trades.

In [ ]:
from src.analysis.asset_inference import AssetInferenceEngine

# Initialize engine using from_directory classmethod
engine = AssetInferenceEngine.from_directory(RAW_PATH)

# Count loaded benchmarks
n_benchmarks = len(engine.benchmarks)
print(f"Loaded {n_benchmarks} benchmarks")

# Group benchmarks by asset class from metadata
asset_classes = {}
for name, meta in engine.bench_meta.items():
    asset_class = meta.get('asset_class', 'unknown')
    if asset_class not in asset_classes:
        asset_classes[asset_class] = []
    asset_classes[asset_class].append(name)

print(f"\nBenchmarks by asset class:")
for asset_class, benchmarks in sorted(asset_classes.items()):
    print(f"  {asset_class}: {len(benchmarks)} benchmarks")

In [ ]:
# Infer assets for a sample of algorithms
# Use more algorithms for better regime detection
sample_size = min(200, len(returns_matrix.columns))
np.random.seed(42)
sample_cols = np.random.choice(returns_matrix.columns, sample_size, replace=False)
returns_sample = returns_matrix[sample_cols]

print(f"Inferring assets for {sample_size} algorithms...")
asset_inferences = {}
for i, algo_id in enumerate(sample_cols):
    if (i + 1) % 50 == 0:
        print(f"  Progress: {i+1}/{sample_size}")
    
    algo_returns = returns_sample[algo_id].dropna()
    if len(algo_returns) > 50:
        try:
            # engine.infer() expects:
            #   algo_daily: DataFrame with 'close' column
            #   algo_returns: Series (optional)
            # We construct a synthetic price series from returns
            close_prices = (1 + algo_returns).cumprod()
            algo_daily = pd.DataFrame({'close': close_prices})
            
            inference = engine.infer(algo_daily, algo_returns)
            asset_inferences[algo_id] = inference
        except Exception as e:
            pass  # Skip algorithms that fail inference

print(f"\nSuccessfully inferred assets for {len(asset_inferences)} algorithms")

In [ ]:
# Summarize asset class distribution
asset_class_counts = pd.Series([inf.asset_class for inf in asset_inferences.values()]).value_counts()
print("Asset Class Distribution:")
print(asset_class_counts)

In [ ]:
# Visualize asset class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = plt.cm.Set3(np.linspace(0, 1, len(asset_class_counts)))
axes[0].pie(asset_class_counts.values, labels=asset_class_counts.index, 
            autopct='%1.1f%%', colors=colors)
axes[0].set_title('Inferred Asset Class Distribution')

# Bar chart
asset_class_counts.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Algorithms per Asset Class')
axes[1].set_xlabel('Asset Class')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 2. Investment Clock Category Mapping

Map algorithms to Investment Clock categories:
- **Cyclical**: Tech, Materials, Industrials, Consumer Discretionary
- **Defensive**: Utilities, Consumer Staples, Healthcare
- **High Yield**: Corporate credit, junk bonds
- **Government**: Treasuries, sovereign bonds
- **Commodities**: Gold, Oil, Agricultural
- **Forex**: Currency pairs

In [ ]:
from src.analysis.latent_regime_inference import InvestmentClockRegimeInference

clock = InvestmentClockRegimeInference(rolling_window=21, vol_window=63)

# Map algorithms to categories
category_mapping = clock._map_algos_to_categories(asset_inferences)

# Count by category
category_counts = pd.Series(category_mapping).value_counts()
print("Investment Clock Category Distribution:")
print(category_counts)

In [ ]:
# Visualize category mapping
fig, ax = plt.subplots(figsize=(10, 6))

category_colors = {
    'cyclical': 'green',
    'defensive': 'red',
    'high_yield': 'orange',
    'government': 'blue',
    'equity_index': 'purple',
    'commodities': 'brown',
    'forex': 'cyan',
    'other': 'gray',
}

colors = [category_colors.get(cat, 'gray') for cat in category_counts.index]
category_counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.set_title('Algorithms by Investment Clock Category')
ax.set_xlabel('Category')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 3. Asset Class Performance

Compute rolling performance by asset class category.

In [5]:
# Filter to algorithms with inferences
algos_with_inference = list(asset_inferences.keys())
returns_for_clock = returns_sample[algos_with_inference]

# Compute asset class performance
performance = clock._compute_asset_class_performance(returns_for_clock, category_mapping)

print("Asset Class Performance computed.")
print(f"Categories: {list(performance.returns.columns)}")

NameError: name 'asset_inferences' is not defined

In [ ]:
# Plot asset class cumulative returns
fig, ax = plt.subplots(figsize=(14, 7))

for col in performance.returns.columns:
    if col != 'other':
        cumret = (1 + performance.returns[col].fillna(0)).cumprod()
        ax.plot(cumret.index, cumret.values, label=col, linewidth=1.5)

ax.axhline(y=1, color='black', linestyle='--', linewidth=0.5)
ax.set_title('Cumulative Returns by Asset Class Category')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Rolling returns by asset class
window = 21  # ~1 month
rolling_returns = performance.returns.rolling(window).mean() * 252  # Annualized

fig, ax = plt.subplots(figsize=(14, 7))

for col in rolling_returns.columns:
    if col != 'other':
        ax.plot(rolling_returns.index, rolling_returns[col].values, 
                label=col, linewidth=1, alpha=0.8)

ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
ax.set_title(f'Rolling {window}-Day Annualized Returns by Asset Class')
ax.set_xlabel('Date')
ax.set_ylabel('Annualized Return')
ax.legend(loc='upper left')
ax.set_ylim(-1, 1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Regime Indicators

Build the regime indicators from asset class performance.

In [ ]:
# Build regime indicators
indicators = clock._build_regime_indicators(performance)

print("Regime Indicators:")
print(f"  cyclical_vs_defensive: {indicators.cyclical_vs_defensive.dropna().shape[0]} observations")
print(f"  credit_spread: {indicators.credit_spread.dropna().shape[0]} observations")
print(f"  equity_vs_bonds: {indicators.equity_vs_bonds.dropna().shape[0]} observations")
print(f"  volatility_level: {indicators.volatility_level.dropna().shape[0]} observations")
print(f"  correlation_level: {indicators.correlation_level.dropna().shape[0]} observations")
print(f"  momentum_breadth: {indicators.momentum_breadth.dropna().shape[0]} observations")

In [ ]:
# Visualize all indicators
fig, axes = plt.subplots(6, 1, figsize=(16, 16), sharex=True)

# 1. Cyclical vs Defensive
ax = axes[0]
data = indicators.cyclical_vs_defensive.dropna()
ax.plot(data.index, data.values, 'b-', linewidth=1)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
ax.fill_between(data.index, 0, data.values, where=data.values > 0, alpha=0.3, color='green', label='Cyclicals winning')
ax.fill_between(data.index, 0, data.values, where=data.values < 0, alpha=0.3, color='red', label='Defensives winning')
ax.set_title('1. Cyclical vs Defensive Spread')
ax.set_ylabel('Spread')
ax.legend(loc='upper right')

# 2. Credit Spread
ax = axes[1]
data = indicators.credit_spread.dropna()
ax.plot(data.index, data.values, 'orange', linewidth=1)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
ax.fill_between(data.index, 0, data.values, where=data.values > 0, alpha=0.3, color='green', label='HY outperforming')
ax.fill_between(data.index, 0, data.values, where=data.values < 0, alpha=0.3, color='red', label='Gov outperforming')
ax.set_title('2. Credit Spread (HY vs Government)')
ax.set_ylabel('Spread')
ax.legend(loc='upper right')

# 3. Equity vs Bonds
ax = axes[2]
data = indicators.equity_vs_bonds.dropna()
ax.plot(data.index, data.values, 'purple', linewidth=1)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
ax.fill_between(data.index, 0, data.values, where=data.values > 0, alpha=0.3, color='green', label='Equity outperforming')
ax.fill_between(data.index, 0, data.values, where=data.values < 0, alpha=0.3, color='red', label='Bonds outperforming')
ax.set_title('3. Equity vs Bonds Spread')
ax.set_ylabel('Spread')
ax.legend(loc='upper right')

# 4. Volatility Level
ax = axes[3]
data = indicators.volatility_level.dropna()
ax.plot(data.index, data.values, 'red', linewidth=1)
mean_vol = data.mean()
ax.axhline(y=mean_vol, color='black', linestyle='--', linewidth=0.5, label=f'Mean: {mean_vol:.2%}')
ax.fill_between(data.index, mean_vol, data.values, where=data.values > mean_vol, alpha=0.3, color='red', label='High vol')
ax.fill_between(data.index, mean_vol, data.values, where=data.values < mean_vol, alpha=0.3, color='green', label='Low vol')
ax.set_title('4. Cross-Sectional Volatility')
ax.set_ylabel('Volatility')
ax.legend(loc='upper right')

# 5. Correlation Level
ax = axes[4]
data = indicators.correlation_level.dropna()
ax.plot(data.index, data.values, 'brown', linewidth=1)
mean_corr = data.mean()
ax.axhline(y=mean_corr, color='black', linestyle='--', linewidth=0.5, label=f'Mean: {mean_corr:.2f}')
ax.set_title('5. Cross-Sectional Correlation')
ax.set_ylabel('Correlation')
ax.legend(loc='upper right')

# 6. Momentum Breadth
ax = axes[5]
data = indicators.momentum_breadth.dropna()
ax.plot(data.index, data.values, 'cyan', linewidth=1)
ax.axhline(y=0.5, color='black', linestyle='--', linewidth=0.5)
ax.fill_between(data.index, 0.5, data.values, where=data.values > 0.5, alpha=0.3, color='green', label='Bullish')
ax.fill_between(data.index, 0.5, data.values, where=data.values < 0.5, alpha=0.3, color='red', label='Bearish')
ax.set_title('6. Momentum Breadth (% of asset classes with positive momentum)')
ax.set_ylabel('Breadth')
ax.set_xlabel('Date')
ax.set_ylim(0, 1)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

---
## 5. Regime Inference

Use HMM to infer the business cycle phase.

In [ ]:
# Run full regime inference
print("Running Investment Clock regime inference...")
result = clock.infer_regime_from_assets(returns_for_clock, asset_inferences)
print("Done.")

In [ ]:
# Display phase statistics
print("\nBUSINESS CYCLE PHASES DETECTED")
print("=" * 60)

phase_names = {0: 'Recovery', 1: 'Expansion', 2: 'Slowdown', 3: 'Contraction'}
phase_descriptions = {
    0: 'Below-trend but rebounding. Cyclicals lead, credit outperforms.',
    1: 'Healthy growth. Broad equity outperformance, low volatility.',
    2: 'Peak growth, decelerating. Defensives start leading.',
    3: 'Declining activity. Safe havens dominate, high volatility.',
}

for phase, stats in sorted(result.phase_statistics.items()):
    name = phase_names.get(phase, f'Phase {phase}')
    desc = phase_descriptions.get(phase, '')
    print(f"\n{phase}. {name}")
    print(f"   {desc}")
    print(f"   Days: {stats['n_days']} ({stats['pct_time']:.1f}% of time)")
    print(f"   Indicators:")
    print(f"     Cyclical vs Defensive: {stats['avg_cyclical_vs_defensive']:.4f}")
    print(f"     Credit Spread: {stats['avg_credit_spread']:.4f}")
    print(f"     Volatility: {stats['avg_volatility']:.2%}")
    print(f"     Correlation: {stats['avg_correlation']:.2f}")
    print(f"     Momentum Breadth: {stats['avg_momentum_breadth']:.1%}")

In [ ]:
# Visualize regime over time with indicators
fig, axes = plt.subplots(5, 1, figsize=(16, 14), sharex=True)

colors = {0: 'green', 1: 'blue', 2: 'orange', 3: 'red'}
phase_names = {0: 'Recovery', 1: 'Expansion', 2: 'Slowdown', 3: 'Contraction'}

# 1. Regime labels
ax = axes[0]
regime_labels = result.regime_labels
for phase in sorted(regime_labels.unique()):
    mask = regime_labels == phase
    ax.fill_between(regime_labels.index, 0, 1, where=mask, 
                   alpha=0.7, color=colors.get(phase, 'gray'),
                   label=phase_names.get(phase, f'Phase {phase}'))
ax.set_title('Business Cycle Regime')
ax.set_ylabel('Regime')
ax.legend(loc='upper right', ncol=4)
ax.set_ylim(0, 1)
ax.set_yticks([])

# 2. Regime probabilities
ax = axes[1]
for i in range(4):
    col = f'phase_{i}'
    if col in result.regime_probabilities.columns:
        ax.plot(result.regime_probabilities.index, result.regime_probabilities[col].values,
               color=colors[i], label=phase_names[i], linewidth=1, alpha=0.7)
ax.set_title('Regime Probabilities')
ax.set_ylabel('Probability')
ax.legend(loc='upper right', ncol=4)
ax.set_ylim(0, 1)

# 3. Cyclical vs Defensive with regime coloring
ax = axes[2]
ind = result.indicators
for phase in sorted(regime_labels.unique()):
    mask = regime_labels == phase
    data = ind.cyclical_vs_defensive.copy()
    data[~mask] = np.nan
    ax.fill_between(data.index, 0, data.values, alpha=0.5, color=colors.get(phase, 'gray'))
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
ax.set_title('Cyclical vs Defensive Spread (colored by regime)')
ax.set_ylabel('Spread')

# 4. Volatility with regime coloring
ax = axes[3]
for phase in sorted(regime_labels.unique()):
    mask = regime_labels == phase
    data = ind.volatility_level.copy()
    data[~mask] = np.nan
    ax.plot(data.index, data.values, color=colors.get(phase, 'gray'), linewidth=1)
ax.axhline(y=ind.volatility_level.mean(), color='black', linestyle='--', linewidth=0.5)
ax.set_title('Volatility Level (colored by regime)')
ax.set_ylabel('Volatility')

# 5. Momentum breadth with regime coloring
ax = axes[4]
for phase in sorted(regime_labels.unique()):
    mask = regime_labels == phase
    data = ind.momentum_breadth.copy()
    data[~mask] = np.nan
    ax.fill_between(data.index, 0.5, data.values, alpha=0.5, color=colors.get(phase, 'gray'))
ax.axhline(y=0.5, color='black', linestyle='--', linewidth=0.5)
ax.set_title('Momentum Breadth (colored by regime)')
ax.set_ylabel('Breadth')
ax.set_xlabel('Date')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Transition matrix heatmap
fig, ax = plt.subplots(figsize=(10, 8))

phase_labels = ['Recovery', 'Expansion', 'Slowdown', 'Contraction']
trans_matrix = result.transition_matrix

sns.heatmap(trans_matrix, annot=True, fmt='.1%', cmap='Blues',
            xticklabels=phase_labels, yticklabels=phase_labels, ax=ax,
            vmin=0, vmax=1)
ax.set_title('Regime Transition Probabilities\n(Probability of moving from row to column)', fontsize=12)
ax.set_xlabel('To Phase', fontsize=11)
ax.set_ylabel('From Phase', fontsize=11)

plt.tight_layout()
plt.show()

# Interpret transition matrix
print("\nTransition Matrix Interpretation:")
for i, from_phase in enumerate(phase_labels):
    max_j = np.argmax(trans_matrix[i])
    if max_j != i:
        print(f"  {from_phase} most likely transitions to {phase_labels[max_j]} ({trans_matrix[i, max_j]:.1%})")
    else:
        print(f"  {from_phase} is persistent ({trans_matrix[i, i]:.1%} stay probability)")

---
## 6. Regime Characteristics Summary

Compare indicator values across regimes.

In [ ]:
# Create summary dataframe
summary_data = []
for phase, stats in sorted(result.phase_statistics.items()):
    summary_data.append({
        'Phase': phase_names.get(phase, f'Phase {phase}'),
        'Days': stats['n_days'],
        '% Time': stats['pct_time'],
        'Cyclical vs Defensive': stats['avg_cyclical_vs_defensive'],
        'Credit Spread': stats['avg_credit_spread'],
        'Volatility': stats['avg_volatility'],
        'Correlation': stats['avg_correlation'],
        'Momentum Breadth': stats['avg_momentum_breadth'],
    })

summary_df = pd.DataFrame(summary_data).set_index('Phase')
print("Regime Characteristics Summary:")
display(summary_df.round(4))

In [ ]:
# Radar chart of regime characteristics
from math import pi

# Normalize indicators for comparison
indicators_to_plot = ['Cyclical vs Defensive', 'Credit Spread', 'Volatility', 'Correlation', 'Momentum Breadth']
normalized = summary_df[indicators_to_plot].copy()

# Normalize to 0-1 range
for col in normalized.columns:
    min_val = normalized[col].min()
    max_val = normalized[col].max()
    if max_val > min_val:
        normalized[col] = (normalized[col] - min_val) / (max_val - min_val)
    else:
        normalized[col] = 0.5

# Create radar chart
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

categories = indicators_to_plot
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Complete the loop

for i, (phase, row) in enumerate(normalized.iterrows()):
    values = row.values.tolist()
    values += values[:1]  # Complete the loop
    color = list(colors.values())[i]
    ax.plot(angles, values, 'o-', linewidth=2, label=phase, color=color)
    ax.fill(angles, values, alpha=0.25, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=10)
ax.set_title('Regime Characteristics Comparison', size=14, y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.show()

---
## 7. Generate Full Report

In [ ]:
# Generate text report
report = clock.generate_report(result)
print(report)

---
## 8. Export Results

Save regime labels for use in RL agent.

In [ ]:
# Save regime labels
from src.utils.paths import ensure_dir
output_path = PROCESSED_PATH / 'phase2'
ensure_dir(output_path)

# Regime labels
result.regime_labels.to_csv(output_path / 'investment_clock_regime_labels.csv')
print(f"Saved regime labels to {output_path / 'investment_clock_regime_labels.csv'}")

# Regime probabilities
result.regime_probabilities.to_parquet(output_path / 'investment_clock_regime_probabilities.parquet')
print(f"Saved regime probabilities to {output_path / 'investment_clock_regime_probabilities.parquet'}")

# Indicators for RL features
indicators_df = pd.DataFrame({
    'cyclical_vs_defensive': result.indicators.cyclical_vs_defensive,
    'credit_spread': result.indicators.credit_spread,
    'equity_vs_bonds': result.indicators.equity_vs_bonds,
    'volatility_level': result.indicators.volatility_level,
    'correlation_level': result.indicators.correlation_level,
    'momentum_breadth': result.indicators.momentum_breadth,
})
indicators_df.to_parquet(output_path / 'investment_clock_indicators.parquet')
print(f"Saved indicators to {output_path / 'investment_clock_indicators.parquet'}")

---
## Summary

The Investment Clock regime inference provides:
1. **Regime labels** for each day (Recovery, Expansion, Slowdown, Contraction)
2. **Regime probabilities** for soft regime membership
3. **Regime indicators** that can be used as RL features:
   - Cyclical vs Defensive spread
   - Credit spread
   - Equity vs Bonds spread
   - Volatility level
   - Correlation level
   - Momentum breadth

These are all computed using **rolling windows** (no lookahead bias) and based on **asset class performance** (cause), not benchmark allocation (effect).